# Computational Theory: SHA-256 Implementation

This notebook contains my solutions to the assessment problems for the
Computational Theory module. The problems are based on the
[Secure Hash Standard (FIPS 180-4)](https://doi.org/10.6028/NIST.FIPS.180-4),
and the overall goal is to build a working implementation of SHA-256 from
first principles.

Each section below corresponds to one problem from the assessment brief.

## Problem 1: Representing SHA-256 Data

Before implementing any part of SHA-256, it's worth thinking about how the
algorithm's inputs, outputs, and intermediate values can be represented in
Python. SHA-256 operates on fixed-size binary data — bits, bytes, and 32-bit
words — so choosing sensible Python and NumPy representations up front makes
the rest of the implementation easier to write, test, and reason about.

In this section I look at how to represent:

- 32-bit words,
- sequences of 32-bit words,
- input messages,
- 512-bit message blocks,
- the final 256-bit hash value,
- big-endian integers.

For each one, I explain why the chosen representation is appropriate and
give a small example demonstrating it.

### Representing 32-bit words

The Secure Hash Standard defines SHA-256's internal operations over 32-bit
words, with arithmetic performed modulo 2³². Python's built-in `int` type is
arbitrary-precision and never wraps around, so using plain integers would
require manually masking the result of every operation with `& 0xFFFFFFFF`.

Instead, I represent each 32-bit word using NumPy's `numpy.uint32` type.
This is a fixed-width unsigned integer type that wraps around automatically
on overflow, matching the modular arithmetic required by the standard.
Using `numpy.uint32` throughout also keeps the code consistent with the
bitwise functions required in Problem 2.

In [1]:
import numpy as np

# A 32-bit word represented using NumPy's fixed-width unsigned integer type
word = np.uint32(0x6a09e667)  # one of SHA-256's initial hash values
print(word, type(word))

# Demonstrate that arithmetic wraps around at 32 bits, as required
max_word = np.uint32(0xFFFFFFFF)
wrapped = max_word + np.uint32(1)
print(f"{max_word:#010x} + 1 = {wrapped:#010x}")  # wraps to 0x00000000

1779033703 <class 'numpy.uint32'>
0xffffffff + 1 = 0x00000000


C:\Users\tomas\AppData\Local\Temp\ipykernel_11488\895404130.py:9: RuntimeWarning: overflow encountered in scalar add
  wrapped = max_word + np.uint32(1)


### Sequences of 32-bit words

Many parts of SHA-256 involve working with several 32-bit words together —
for example, the eight working variables, the 64 round constants, or the
64-word message schedule used in the compression function.

I represent a sequence of 32-bit words as a NumPy array with
`dtype=numpy.uint32`. This ensures every element in the sequence is
constrained to 32 bits (rather than relying on each element being created
correctly by hand), and it allows element-wise operations to be applied
across the whole sequence at once, which will be useful later.

In [2]:
# A sequence of 32-bit words, e.g. the eight SHA-256 initial hash values
initial_hash_values = np.array([
    0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a,
    0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19
], dtype=np.uint32)

print(initial_hash_values)
print(initial_hash_values.dtype)

# Element-wise operation applied across the whole sequence at once
shifted = initial_hash_values + np.uint32(1)
print(shifted)


[1779033703 3144134277 1013904242 2773480762 1359893119 2600822924
  528734635 1541459225]
uint32
[1779033704 3144134278 1013904243 2773480763 1359893120 2600822925
  528734636 1541459226]


### Input messages

The Secure Hash Standard treats a message as a sequence of bits (or,
equivalently, a sequence of 8-bit bytes). This means SHA-256 does not
operate on text directly — any text input must first be converted to bytes
using some encoding (I use UTF-8, the standard default).

I represent an input message using Python's built-in `bytes` type. This is
immutable, maps directly onto the standard's notion of a message, and
matches the input type expected by Python's own `hashlib.sha256`, which I
use later to verify my implementation.

In [3]:
# A message represented as bytes, converted explicitly from text
text_message = "abc"
message = text_message.encode("utf-8")

print(message, type(message))
print(len(message), "bytes =", len(message) * 8, "bits")

# A message can also be given directly as bytes, without text at all
raw_message = b"\x00\x01\x02\xff"
print(raw_message, len(raw_message), "bytes")

b'abc' <class 'bytes'>
3 bytes = 24 bits
b'\x00\x01\x02\xff' 4 bytes


### 512-bit message blocks

SHA-256 processes a message in 512-bit (64-byte) blocks. Each block is
further divided into sixteen 32-bit words for use in the compression
function, so a block is naturally represented using the same
sequence-of-words representation chosen above: a NumPy array of sixteen
`numpy.uint32` values.

To build a block from raw bytes, the 64 bytes are split into sixteen
groups of 4 bytes each, and each group is interpreted as a big-endian
32-bit unsigned integer — the byte order required by the standard (this is
covered in more detail in the "big-endian integers" subsection below).

This example uses a block of 64 bytes directly, rather than a full message,
since the padding and splitting of arbitrary-length messages into blocks is
handled separately in Problem 4.

In [ ]:
# A 512-bit (64-byte) block, built here directly for demonstration purposes
block_bytes = bytes(range(64))  # 64 arbitrary bytes, 0x00 to 0x3f

# Split into sixteen 4-byte groups, each read as a big-endian uint32
block_words = np.array(
    [int.from_bytes(block_bytes[i:i + 4], byteorder="big") for i in range(0, 64, 4)],
    dtype=np.uint32
)

print(block_words)
print(block_words.dtype, block_words.shape)
print([hex(w) for w in block_words])

[     66051   67438087  134810123  202182159  269554195  336926231
  404298267  471670303  539042339  606414375  673786411  741158447
  808530483  875902519  943274555 1010646591]
uint32 (16,)
['0x10203', '0x4050607', '0x8090a0b', '0xc0d0e0f', '0x10111213', '0x14151617', '0x18191a1b', '0x1c1d1e1f', '0x20212223', '0x24252627', '0x28292a2b', '0x2c2d2e2f', '0x30313233', '0x34353637', '0x38393a3b', '0x3c3d3e3f']


## Problem 2: SHA-256 Bitwise Operations

## Problem 3: Generating the SHA-256 Constants

## Problem 4: Padding and Parsing Messages

## Problem 5: The SHA-256 Compression Function

## Problem 6: Complete SHA-256

***

**End**